<a href="https://colab.research.google.com/github/w50948244-alt/portafoliowr/blob/main/entrenar_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Entrenamiento en Colab (GPU gratis)

**Antes de correr:** Ve a `Entorno de ejecución > Cambiar tipo de entorno de ejecución` y selecciona **GPU (T4)**.

Este notebook espera que subas un `.zip` con tu carpeta `data/` (la que ya organizaste con `fetch_hf_pets.py`, `fetch_hf_faces.py`, `fetch_hf_illustration.py`), con esta estructura:
```
data/
  train/0_Ilustracion ... 8_Nina
  val/0_Ilustracion ... 8_Nina
```

In [ ]:
!pip install -q safetensors
import torch
print('GPU disponible:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

GPU disponible: True
Device: Tesla T4


In [ ]:
!ls data/train

0_Ilustracion  2_Perro	      4_HombreJoven  6_MujerMayor  8_Nina
1_Gato	       3_HombreMayor  5_Nino	     7_MujerJoven


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Subir tu data.zip
Comprime tu carpeta `data/` en tu PC (clic derecho > Enviar a > Carpeta comprimida) y súbela aquí.

In [5]:
from google.colab import files

uploaded = files.upload()  # selecciona tu data.zip

import zipfile
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')
print('Descomprimido. Contenido de data/:')
!ls data/train

Saving data.zip to data.zip
Descomprimido. Contenido de data/:
0_Ilustracion  2_Perro	      4_HombreJoven  6_MujerMayor  8_Nina
1_Gato	       3_HombreMayor  5_Nino	     7_MujerJoven


## 2. Arquitectura (NO MODIFICAR)

In [6]:
import torch.nn as nn
import torch.nn.functional as F


class Stem(nn.Module):
    def __init__(self):
        super(Stem, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=2),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )

    def forward(self, x):
        x = self.conv(x)
        return x


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(inplace=True),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.shortcut = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        )
        self.act = nn.LeakyReLU(inplace=True)

    def forward(self, x):
        identity = self.shortcut(x)
        x = self.conv1(x)
        x = self.conv2(x)
        x += identity
        return self.act(x)


class FromZero(nn.Module):
    def __init__(self, num_classes=10):
        super(FromZero, self).__init__()
        self.stem = nn.Sequential(Stem())
        self.layer1 = nn.Sequential(ResidualBlock(64, 64), ResidualBlock(64, 64))
        self.layer2 = nn.Sequential(ResidualBlock(64, 128, stride=2), ResidualBlock(128, 128))
        self.layer3 = nn.Sequential(ResidualBlock(128, 256, stride=2), ResidualBlock(256, 256))
        self.layer4 = nn.Sequential(ResidualBlock(256, 512, stride=2), ResidualBlock(512, 512), nn.Dropout(0.2))
        self.flatten = nn.Flatten()
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

## 3. Datos, entrenamiento y exportación

In [13]:
import copy
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from safetensors.torch import save_file

CLASSES_ESPERADAS = [
    '0_Ilustracion', '1_Gato', '2_Perro', '3_HombreMayor', '4_HombreJoven',
    '5_Nino', '6_MujerMayor', '7_MujerJoven', '8_Nina',
]

IMG_SIZE = 128
BATCH_SIZE = 64
EPOCHS = 60
LR = 1e-3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Usando device:', device)

train_tf = transforms.Compose([
       transforms.Resize((int(IMG_SIZE * 1.15), int(IMG_SIZE * 1.15))),
       transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
       transforms.RandomHorizontalFlip(),
       transforms.RandomRotation(15),
       transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
       transforms.ToTensor(),
   ])
   val_tf = transforms.Compose([
       transforms.Resize((IMG_SIZE, IMG_SIZE)),
       transforms.ToTensor(),
   ])

train_ds = datasets.ImageFolder('data/train', transform=train_tf)
val_ds = datasets.ImageFolder('data/val', transform=val_tf)

assert train_ds.classes == CLASSES_ESPERADAS, f'Orden incorrecto: {train_ds.classes}'
print('Clases en orden correcto:', train_ds.classes)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model = FromZero(num_classes=9).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    running_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
            running_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total


best_acc = 0.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step(val_acc)
    print(f'Epoch {epoch:03d}/{EPOCHS} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
          f'val_loss={val_loss:.4f} val_acc={val_acc:.4f} | {time.time()-t0:.1f}s')
    if val_acc > best_acc:
        best_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        state_dict_clean = {k: v.contiguous().cpu() for k, v in best_state.items()}
        save_file(state_dict_clean, '/content/drive/MyDrive/WarlliJanserRomanoGarcia_24MISN2030.safetensors')
        print(f'  -> nuevo mejor modelo (val_acc={best_acc:.4f}) [guardado en Drive]')

print(f'\nMejor val_acc: {best_acc:.4f}')

Usando device: cuda
Clases en orden correcto: ['0_Ilustracion', '1_Gato', '2_Perro', '3_HombreMayor', '4_HombreJoven', '5_Nino', '6_MujerMayor', '7_MujerJoven', '8_Nina']
Epoch 001/60 | train_loss=1.9228 train_acc=0.2424 | val_loss=2.1020 val_acc=0.2500 | 11.7s
  -> nuevo mejor modelo (val_acc=0.2500) [guardado en Drive]
Epoch 002/60 | train_loss=1.6716 train_acc=0.3272 | val_loss=1.8239 val_acc=0.3143 | 10.0s
  -> nuevo mejor modelo (val_acc=0.3143) [guardado en Drive]
Epoch 003/60 | train_loss=1.6096 train_acc=0.3576 | val_loss=1.8578 val_acc=0.2679 | 13.3s
Epoch 004/60 | train_loss=1.5041 train_acc=0.3924 | val_loss=2.1150 val_acc=0.2875 | 12.2s
Epoch 005/60 | train_loss=1.4675 train_acc=0.4027 | val_loss=4.2307 val_acc=0.1857 | 11.6s
Epoch 006/60 | train_loss=1.3645 train_acc=0.4348 | val_loss=1.3368 val_acc=0.4554 | 10.1s
  -> nuevo mejor modelo (val_acc=0.4554) [guardado en Drive]
Epoch 007/60 | train_loss=1.3304 train_acc=0.4500 | val_loss=2.1710 val_acc=0.3696 | 12.5s
Epoch 008

## 4. Exportar y descargar el .safetensors
Cambia `NOMBRE_MATRICULA` por tu nombre y matrícula reales antes de correr esta celda.

In [14]:
NOMBRE_MATRICULA = 'WarlliJanserRomanoGarcia_24MISN2030'

state_dict_clean = {k: v.contiguous().cpu() for k, v in best_state.items()}
out_name = f'{NOMBRE_MATRICULA}.safetensors'
save_file(state_dict_clean, out_name)
print('Guardado:', out_name)

from google.colab import files
files.download(out_name)


Guardado: WarlliJanserRomanoGarcia_24MISN2030.safetensors


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
from google.colab import drive
drive.mount('/content/drive')
!ln -s /content/drive/MyDrive/data data

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
